<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Tabnet_Cow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
!pip install pytorch-tabnet
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

In [47]:
cow='/content/rural_carbon_dataset1.csv'
cow_df=pd.read_csv(cow)
cow_df.head()


,Region,Month,Fertilizer_Usage_kg,Crop_Type,Crop_Area_ha,Livestock_Cows,Livestock_Pigs,Household_Energy_kWh,Renewable_Energy_Fraction,Temperature_C,Rainfall_mm,Carbon_Emission_tCO2,Year,classes
0,R0103,4,106.5,Maize,37.3,0,130,296.6,0.58,23.6,70.7,5.91,2024,Low
1,R0271,11,100.9,Wheat,47.4,95,196,408.8,0.50,24.9,275.5,21.63,2021,High
2,R0107,7,59.4,Maize,28.7,83,120,479.3,0.97,21.8,35.7,9.24,2022,Low
3,R0072,8,122.9,Rice,29.3,53,82,146.3,0.13,30.7,285.5,3.71,2024,Low
4,R0189,2,66.4,Maize,25.7,46,169,233.9,0.90,23.3,76.2,4.37,2020,Low


In [48]:
# Drop unnecessary columns
cow_df = cow_df.drop(['Region', 'Year', 'Month', 'Carbon_Emission_tCO2'], axis=1, errors='ignore')

# Separate target from predictors
target = cow_df['classes']
Features = cow_df.drop(columns=['classes'])

# Identify categorical columns and their indices
categorical_columns = [col for col in Features.columns if Features[col].dtype == 'object' or Features[col].dtype.name == 'category']
cat_idxs = [i for i, f in enumerate(Features.columns) if f in categorical_columns]

# Split the dataset into train, validation, and test sets
X_train_df, X_tmp_df, y_train, y_tmp = train_test_split(Features, target.values, test_size=0.3, random_state=42, stratify=target.values)
X_valid_df, X_test_df, y_valid, y_test = train_test_split(X_tmp_df, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

cat_dims = []
X_train = X_train_df.copy()
X_valid = X_valid_df.copy()
X_test = X_test_df.copy()

# Process categorical columns using LabelEncoder safely
for col in categorical_columns:
    l_enc = LabelEncoder()

    # Fit is applied only to the training data to prevent data leakage
    X_train[col] = l_enc.fit_transform(X_train_df[col].values)

    # Transform is applied to validation and test data while handling unseen categories
    # Unseen categories are mapped to -1 to avoid runtime errors
    X_valid[col] = X_valid_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)
    X_test[col] = X_test_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)

    # Calculate dimensions (adding +1 to account for the unknown category fallback '-1')
    cat_dims.append(len(l_enc.classes_) + 1)

# Convert feature dataframes to Numpy arrays ready for TabNet
X_train = X_train.values
X_valid = X_valid.values
X_test = X_test.values

# Output configuration summaries
print("Data preprocessing completed and data leakage resolved successfully!")
print(f"Categorical column dimensions (cat_dims): {cat_dims}")
print(f"Categorical column indexes (cat_idxs): {cat_idxs}")




Data preprocessing completed and data leakage resolved successfully!
Categorical column dimensions (cat_dims): [5]
Categorical column indexes (cat_idxs): [1]


In [49]:
#  Define the TabNet model configuration
clf = TabNetClassifier(
    cat_idxs=cat_idxs,               # Positions/indexes of categorical columns
    cat_dims=cat_dims,               # Dimensions (number of unique classes) for each categorical column
    cat_emb_dim=1,                   # Embedding dimension size for each categorical variable (can be increased for high-cardinality)
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax'            # Provides better interpretability by enforcing sparse, clear feature selection
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [50]:
# Model training execution
clf.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['accuracy', 'logloss'], # Evaluation metrics used to monitor training performance
    max_epochs=100,
    patience=15,                    # Early stopping threshold if validation performance does not improve for 15 epochs
    batch_size=128,
    virtual_batch_size=16,          # Technique used in TabNet (Ghost Batch Normalization) to stabilize training
    num_workers=0,
    drop_last=False
)

epoch 0  | loss: 1.32196 | train_accuracy: 0.44095 | train_logloss: 3.24316 | valid_accuracy: 0.42444 | valid_logloss: 3.09507 |  0:00:02s
epoch 1  | loss: 1.05915 | train_accuracy: 0.4919  | train_logloss: 2.38332 | valid_accuracy: 0.48667 | valid_logloss: 2.54611 |  0:00:04s
epoch 2  | loss: 1.00666 | train_accuracy: 0.42762 | train_logloss: 1.84099 | valid_accuracy: 0.43778 | valid_logloss: 1.85981 |  0:00:06s
epoch 3  | loss: 0.97206 | train_accuracy: 0.50952 | train_logloss: 1.80024 | valid_accuracy: 0.5     | valid_logloss: 1.76252 |  0:00:08s
epoch 4  | loss: 0.98845 | train_accuracy: 0.52048 | train_logloss: 1.16353 | valid_accuracy: 0.53333 | valid_logloss: 1.2016  |  0:00:09s
epoch 5  | loss: 0.96738 | train_accuracy: 0.51667 | train_logloss: 1.13564 | valid_accuracy: 0.53333 | valid_logloss: 1.1556  |  0:00:11s
epoch 6  | loss: 0.94614 | train_accuracy: 0.4881  | train_logloss: 1.08564 | valid_accuracy: 0.50889 | valid_logloss: 1.08744 |  0:00:13s
epoch 7  | loss: 0.94    | 

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [51]:
# Model evaluation and prediction on unseen test data
predictions = clf.predict(X_test)
print(f"Test Accuracy: {np.mean(predictions == y_test):.4f}")

Test Accuracy: 0.5978


In [52]:
# Extract the global relative importance for each feature across the entire dataset
feature_importances = clf.feature_importances_

for feature_name, importance in zip(Features.columns, feature_importances):
    print(f"Feature: {feature_name} -> Importance: {importance:.4f}")

# Extract the interpretability masks (attention masks) to inspect feature weights for individual samples
explain_matrix, masks = clf.explain(X_test)
# You can plot these masks using matplotlib to visualize the attention distribution visually

Feature: Fertilizer_Usage_kg -> Importance: 0.0204
Feature: Crop_Type -> Importance: 0.0003
Feature: Crop_Area_ha -> Importance: 0.1231
Feature: Livestock_Cows -> Importance: 0.3838
Feature: Livestock_Pigs -> Importance: 0.2056
Feature: Household_Energy_kWh -> Importance: 0.0235
Feature: Renewable_Energy_Fraction -> Importance: 0.1806
Feature: Temperature_C -> Importance: 0.0622
Feature: Rainfall_mm -> Importance: 0.0007


#optimized TabNet model

In [53]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

# ==========================================
# 1. Class Merging & Transformation
# ==========================================
# Merge 'Low' and 'Safe' to create a more balanced class distribution
category_mapping = {
    'Low': 'Low_Safe',
    'Safe': 'Low_Safe'
}
cow_df['classes'] = cow_df['classes'].replace(category_mapping)

# Separate target from predictors
target = cow_df['classes']
Features = cow_df.drop(columns=['classes'])

# Identify categorical and numerical columns dynamically
categorical_columns = [col for col in Features.columns if Features[col].dtype == 'object' or Features[col].dtype.name == 'category']
numerical_columns = [col for col in Features.columns if col not in categorical_columns]

cat_idxs = [i for i, f in enumerate(Features.columns) if f in categorical_columns]

# Split the dataset safely into train, validation, and test sets
X_train_df, X_tmp_df, y_train, y_tmp = train_test_split(Features, target.values, test_size=0.3, random_state=42, stratify=target.values)
X_valid_df, X_test_df, y_valid, y_test = train_test_split(X_tmp_df, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

X_train = X_train_df.copy()
X_valid = X_valid_df.copy()
X_test = X_test_df.copy()

# ==========================================
# 2. StandardScaler & LabelEncoder Application
# ==========================================
# Apply StandardScaler to numerical features safely to avoid data leakage
if len(numerical_columns) > 0:
    scaler = StandardScaler()
    X_train[numerical_columns] = scaler.fit_transform(X_train_df[numerical_columns])
    X_valid[numerical_columns] = scaler.transform(X_valid_df[numerical_columns])
    X_test[numerical_columns] = scaler.transform(X_test_df[numerical_columns])

# Apply LabelEncoder to categorical features safely
cat_dims = []
for col in categorical_columns:
    l_enc = LabelEncoder()
    X_train[col] = l_enc.fit_transform(X_train_df[col].values)

    # Handle any potential unseen categories during transformation
    X_valid[col] = X_valid_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)
    X_test[col] = X_test_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)

    cat_dims.append(len(l_enc.classes_) + 1)

# Convert fully preprocessed DataFrames to NumPy arrays for TabNet
X_train = X_train.values
X_valid = X_valid.values
X_test = X_test.values

# ==========================================
# 3. Optimized TabNet Initialization (clf77)
# ==========================================
clf77 = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=1,                   # Embedding dimension size for each categorical variable

    # Structural reduction to prevent overfitting on 3000 rows
    n_d=4,                           # Reduced width of the decision prediction layer
    n_a=4,                           # Reduced width of the attention embedding layer
    n_steps=2,                       # Fewer decision steps to enforce generalization
    lambda_sparse=1e-2,              # Higher sparsity penalty to force selection of only critical features

    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=5e-3),  # Lower learning rate for stable convergence
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size": 20, "gamma": 0.8},
    mask_type='sparsemax'            # Sparse attention mechanism for cleaner feature selection
)

# ==========================================
# 4. Model Training Execution
# ==========================================
clf77.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['accuracy', 'logloss'], # Metrics to monitor performance and overfitting
    max_epochs=200,                  # Increased epochs to compensate for a smaller learning rate
    patience=15,                     # Higher patience to allow the model to escape local minima
    batch_size=64,                   # Smaller batch size to increase weight updates per epoch
    virtual_batch_size=8,            # Smaller virtual batch for stable Ghost Batch Normalization
    num_workers=0,
    drop_last=False
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.18001 | train_accuracy: 0.4719  | train_logloss: 1.05792 | valid_accuracy: 0.43111 | valid_logloss: 1.09428 |  0:00:03s
epoch 1  | loss: 1.06108 | train_accuracy: 0.50857 | train_logloss: 1.01143 | valid_accuracy: 0.51778 | valid_logloss: 1.02922 |  0:00:09s
epoch 2  | loss: 1.01159 | train_accuracy: 0.53667 | train_logloss: 0.96657 | valid_accuracy: 0.54444 | valid_logloss: 0.9729  |  0:00:13s
epoch 3  | loss: 0.97047 | train_accuracy: 0.54619 | train_logloss: 0.933   | valid_accuracy: 0.56222 | valid_logloss: 0.9204  |  0:00:17s
epoch 4  | loss: 0.95107 | train_accuracy: 0.55238 | train_logloss: 0.91439 | valid_accuracy: 0.56889 | valid_logloss: 0.89461 |  0:00:21s
epoch 5  | loss: 0.92458 | train_accuracy: 0.56238 | train_logloss: 0.88825 | valid_accuracy: 0.57333 | valid_logloss: 0.85841 |  0:00:23s
epoch 6  | loss: 0.91161 | train_accuracy: 0.56429 | train_logloss: 0.88776 | valid_accuracy: 0.6     | valid_logloss: 0.85661 |  0:00:24s
epoch 7  | loss: 0.89878 | 

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [54]:
# Model evaluation and prediction on unseen test data
predictions = clf77.predict(X_test)
print(f"Test Accuracy: {np.mean(predictions == y_test):.4f}")

Test Accuracy: 0.6622


#2nd: With feature engineering

In [56]:
#Feature engineering
cow_2=cow_df.drop(['Region','Year','Month','Carbon_Emission_tCO2'], axis=1, errors='ignore')

# 4. Feature Engineering
cow_weight = 2.0  # Approx tons of CO2e per cow/year
pig_weight = 0.3  # Approx tons of CO2e per pig/year

cow_2['Weighted_Livestock_Impact'] = (cow_2['Livestock_Cows'] * cow_weight) + (cow_df['Livestock_Pigs'] * pig_weight)
# 2. Advanced Density Feature: Connect the weighted impact with the land area
# cow_df['Livestock_Impact_per_Ha'] = cow_df['Weighted_Livestock_Impact'] / (cow_df['Crop_Area_ha'] + 1e-5)

cow_2['Fert_x_Area'] = cow_2['Fertilizer_Usage_kg'] / cow_2['Crop_Area_ha']
cow_2['Renew_x_Energy'] = cow_2['Renewable_Energy_Fraction'] * cow_2['Household_Energy_kWh']
cow_2['energy_intensity'] = cow_2['Household_Energy_kWh'] / (cow_2['Crop_Area_ha'] + 1e-5)
cow_2=cow_2.drop(['Livestock_Pigs','Livestock_Cows','Renewable_Energy_Fraction','Fertilizer_Usage_kg'], axis=1)

# print a sample of dataset after preprocessing
cow_2.head()

,Crop_Type,Crop_Area_ha,Household_Energy_kWh,Temperature_C,Rainfall_mm,classes,Weighted_Livestock_Impact,Fert_x_Area,Renew_x_Energy,energy_intensity
0,Maize,37.3,296.6,23.6,70.7,Low_Safe,39.0,2.855228,172.028,7.951740
1,Wheat,47.4,408.8,24.9,275.5,High,248.8,2.128692,204.400,8.624471
2,Maize,28.7,479.3,21.8,35.7,Low_Safe,202.0,2.069686,464.921,16.700343
3,Rice,29.3,146.3,30.7,285.5,Low_Safe,130.6,4.194539,19.019,4.993172
4,Maize,25.7,233.9,23.3,76.2,Low_Safe,142.7,2.583658,210.510,9.101164


In [62]:
# 1. Class Merging & Transformation (Dataset 2)
# ==========================================
# Merge 'Low' and 'Safe' into 'Low_Safe' for dataset 2 to balance distribution
category2_mapping = {
    'Low': 'Low_Safe',
    'Safe': 'Low_Safe'
}
cow_2['classes'] = cow_2['classes'].replace(category2_mapping)

# Separate target from predictors for dataset 2
target2 = cow_2['classes']
Features2 = cow_2.drop(columns=['classes'])

# Identify categorical and numerical columns dynamically for dataset 2
categorical2_columns = [col for col in Features2.columns if Features2[col].dtype == 'object' or Features2[col].dtype.name == 'category']
numerical2_columns = [col for col in Features2.columns if col not in categorical2_columns]

cat2_idxs = [i for i, f in enumerate(Features2.columns) if f in categorical2_columns]

# Split the second dataset into train, validation, and test sets safely
X2_train_df, X2_tmp_df, y2_train, y2_tmp = train_test_split(Features2, target2.values, test_size=0.3, random_state=42, stratify=target2.values)
X2_valid_df, X2_test_df, y2_valid, y2_test = train_test_split(X2_tmp_df, y2_tmp, test_size=0.5, random_state=42, stratify=y2_tmp)

X2_train = X2_train_df.copy()
X2_valid = X2_valid_df.copy()
X2_test = X2_test_df.copy()

# ==========================================
# 2. StandardScaler & LabelEncoder Application
# ==========================================
# Apply StandardScaler to numerical features safely to avoid data leakage
if len(numerical2_columns) > 0:
    scaler2 = StandardScaler()
    X2_train[numerical2_columns] = scaler2.fit_transform(X2_train_df[numerical2_columns])
    X2_valid[numerical2_columns] = scaler2.transform(X2_valid_df[numerical2_columns])
    X2_test[numerical2_columns] = scaler2.transform(X2_test_df[numerical2_columns])

# Apply LabelEncoder to categorical features safely
cat2_dims = []
for col in categorical2_columns:
    l_enc = LabelEncoder()
    X2_train[col] = l_enc.fit_transform(X2_train_df[col].values)

    # Handle potential unseen categories during transformation by mapping to -1
    X2_valid[col] = X2_valid_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)
    X2_test[col] = X2_test_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)

    cat2_dims.append(len(l_enc.classes_) + 1)

# Convert preprocessed DataFrames to NumPy arrays for TabNet
X2_train = X2_train.values
X2_valid = X2_valid.values
X2_test = X2_test.values

print("Data preprocessing with scaling and encoding completed successfully for dataset 2!")


Data preprocessing with scaling and encoding completed successfully for dataset 2!


In [66]:
# 5. Define the TabNet model configuration for dataset 2
clf2 = TabNetClassifier(
    cat_idxs=cat2_idxs,              # Positions/indexes of categorical columns for dataset 2
    cat_dims=cat2_dims,              # Dimensions (number of unique classes) for each categorical column in dataset 2
    cat_emb_dim=1,                   # Embedding dimension size for each categorical variable (can be increased for high-cardinality)
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax'            # Provides better interpretability by enforcing sparse, clear feature selection
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [67]:
# 6. Model training execution for dataset 2
clf2.fit(
    X_train=X2_train, y_train=y2_train,
    eval_set=[(X2_train, y2_train), (X2_valid, y2_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['accuracy', 'logloss'], # Evaluation metrics used to monitor training performance
    max_epochs=100,
    patience=15,                    # Early stopping threshold if validation performance does not improve for 15 epochs
    batch_size=128,
    virtual_batch_size=16,          # Technique used in TabNet (Ghost Batch Normalization) to stabilize training
    num_workers=0,
    drop_last=False
)

epoch 0  | loss: 1.13366 | train_accuracy: 0.54714 | train_logloss: 0.97034 | valid_accuracy: 0.54444 | valid_logloss: 0.97495 |  0:00:00s
epoch 1  | loss: 0.94411 | train_accuracy: 0.54429 | train_logloss: 0.93036 | valid_accuracy: 0.58444 | valid_logloss: 0.89593 |  0:00:01s
epoch 2  | loss: 0.93136 | train_accuracy: 0.56952 | train_logloss: 0.88966 | valid_accuracy: 0.59111 | valid_logloss: 0.87896 |  0:00:02s
epoch 3  | loss: 0.90094 | train_accuracy: 0.56476 | train_logloss: 0.86508 | valid_accuracy: 0.62444 | valid_logloss: 0.85891 |  0:00:03s
epoch 4  | loss: 0.88515 | train_accuracy: 0.58238 | train_logloss: 0.84613 | valid_accuracy: 0.62889 | valid_logloss: 0.83164 |  0:00:04s
epoch 5  | loss: 0.86934 | train_accuracy: 0.59857 | train_logloss: 0.8279  | valid_accuracy: 0.64    | valid_logloss: 0.80805 |  0:00:05s
epoch 6  | loss: 0.86132 | train_accuracy: 0.6019  | train_logloss: 0.82529 | valid_accuracy: 0.64889 | valid_logloss: 0.80082 |  0:00:05s
epoch 7  | loss: 0.85643 | 

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [60]:
# 7. Model evaluation and prediction on unseen test data for dataset 2
predictions = clf2.predict(X2_test)
print(f"Test Accuracy: {np.mean(predictions == y2_test):.4f}")

Test Accuracy: 0.6200


In [61]:
# Extract the global relative importance for each feature across the entire second dataset
feature_importances2 = clf2.feature_importances_

for feature_name, importance in zip(Features2.columns, feature_importances2):
    print(f"Feature: {feature_name} -> Importance: {importance:.4f}")

# Extract the interpretability masks (attention masks) to inspect feature weights for individual samples in dataset 2
explain_matrix, masks = clf2.explain(X2_test)
# You can plot these masks using matplotlib to visualize the attention distribution visually

Feature: Crop_Type -> Importance: 0.0092
Feature: Crop_Area_ha -> Importance: 0.0151
Feature: Household_Energy_kWh -> Importance: 0.0697
Feature: Temperature_C -> Importance: 0.0429
Feature: Rainfall_mm -> Importance: 0.0059
Feature: Weighted_Livestock_Impact -> Importance: 0.3764
Feature: Fert_x_Area -> Importance: 0.0566
Feature: Renew_x_Energy -> Importance: 0.4242
Feature: energy_intensity -> Importance: 0.0000


In [63]:
# 3. Optimized TabNet Initialization (clf77)
# ==========================================
clf277 = TabNetClassifier(
    cat_idxs=cat2_idxs,
    cat_dims=cat2_dims,
    cat_emb_dim=1,                   # Embedding dimension size for each categorical variable

    # Structural reduction to prevent overfitting on 3000 rows
    n_d=4,                           # Reduced width of the decision prediction layer
    n_a=4,                           # Reduced width of the attention embedding layer
    n_steps=2,                       # Fewer decision steps to enforce generalization
    lambda_sparse=1e-2,              # Higher sparsity penalty to force selection of only critical features

    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=5e-3),  # Lower learning rate for stable convergence
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size": 20, "gamma": 0.8},
    mask_type='sparsemax'            # Sparse attention mechanism for cleaner feature selection
)

# ==========================================
# 4. Model Training Execution
# ==========================================
clf277.fit(
    X_train=X2_train, y_train=y2_train,
    eval_set=[(X2_train, y2_train), (X2_valid, y2_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['accuracy', 'logloss'], # Metrics to monitor performance and overfitting
    max_epochs=200,                  # Sufficient epochs to reach optimal convergence
    patience=15,                     # Early stopping criteria to halt training if validation stops improving
    batch_size=64,                   # Smaller batch size to increase weight updates per epoch
    virtual_batch_size=8,            # Smaller virtual batch for stable Ghost Batch Normalization
    num_workers=0,
    drop_last=False
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.19813 | train_accuracy: 0.46    | train_logloss: 1.06574 | valid_accuracy: 0.46222 | valid_logloss: 1.04083 |  0:00:02s
epoch 1  | loss: 1.05069 | train_accuracy: 0.51286 | train_logloss: 1.01117 | valid_accuracy: 0.52444 | valid_logloss: 0.98773 |  0:00:04s
epoch 2  | loss: 1.01194 | train_accuracy: 0.53524 | train_logloss: 0.96891 | valid_accuracy: 0.57556 | valid_logloss: 0.93642 |  0:00:06s
epoch 3  | loss: 0.97648 | train_accuracy: 0.56667 | train_logloss: 0.93664 | valid_accuracy: 0.60222 | valid_logloss: 0.90676 |  0:00:07s
epoch 4  | loss: 0.95578 | train_accuracy: 0.56762 | train_logloss: 0.91578 | valid_accuracy: 0.59778 | valid_logloss: 0.9006  |  0:00:09s
epoch 5  | loss: 0.95545 | train_accuracy: 0.57524 | train_logloss: 0.89564 | valid_accuracy: 0.58667 | valid_logloss: 0.88591 |  0:00:10s
epoch 6  | loss: 0.92033 | train_accuracy: 0.5881  | train_logloss: 0.88082 | valid_accuracy: 0.60444 | valid_logloss: 0.87374 |  0:00:12s
epoch 7  | loss: 0.92035 | 

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [68]:
# 7. Model evaluation and prediction on unseen test data for dataset 2
predictions = clf277.predict(X2_test)
print(f"Test Accuracy: {np.mean(predictions == y2_test):.4f}")

Test Accuracy: 0.6289
